In [57]:
import requests
import pandas as pd

# 請將 'YOUR_API_KEY' 替換為您的環境部 API Key
API_KEY = ''
URL = 'https://data.moenv.gov.tw/api/v2/aqx_p_432'

# 設定請求參數
params = {
    'api_key': API_KEY,
    'limit': 1000,    # 抓取筆數上限
    'format': 'JSON'  # 回傳格式
}

# 發送 GET 請求
response = requests.get(URL, params=params)

if response.status_code == 200:
    data = response.json()
    print("成功取得資料！")
else:
    print(f"請求失敗，狀態碼：{response.status_code}")
    print(response.text)

成功取得資料！


In [58]:
pd.DataFrame(data).columns

Index(['sitename', 'county', 'aqi', 'pollutant', 'status', 'so2', 'co', 'o3',
       'o3_8hr', 'pm10', 'pm2.5', 'no2', 'nox', 'no', 'wind_speed',
       'wind_direc', 'publishtime', 'co_8hr', 'pm2.5_avg', 'pm10_avg',
       'so2_avg', 'longitude', 'latitude', 'siteid'],
      dtype='object')

In [59]:
raw_data = pd.DataFrame(data)

In [60]:
raw_data = raw_data[raw_data.sitename.isin(['桃園', '觀音', '大園', '平鎮', '龍潭'])]

In [61]:
raw_data.columns

Index(['sitename', 'county', 'aqi', 'pollutant', 'status', 'so2', 'co', 'o3',
       'o3_8hr', 'pm10', 'pm2.5', 'no2', 'nox', 'no', 'wind_speed',
       'wind_direc', 'publishtime', 'co_8hr', 'pm2.5_avg', 'pm10_avg',
       'so2_avg', 'longitude', 'latitude', 'siteid'],
      dtype='object')

In [62]:
col = ['sitename', 'publishtime', 'pm2.5', 'pm10', 'o3', 'co', 'so2', 'no2', 'wind_speed', 'wind_direc']
raw_data = raw_data[col]

In [63]:
# import requests
# import pandas as pd

# # 請將 'YOUR_API_KEY' 替換為您的環境部 API Key
# API_KEY = ''
# URL = 'https://data.moenv.gov.tw/api/v2/aqx_p_313'
# # 設定請求參數
# params = {
#     'api_key': API_KEY,
#     'limit': 1000,    # 抓取筆數上限
#     'format': 'JSON'  # 回傳格式
# }

# # 發送 GET 請求
# response = requests.get(URL, params=params)

# if response.status_code == 200:
#     data = response.json()
#     print("成功取得資料！")
# else:
#     print(f"請求失敗，狀態碼：{response.status_code}")
#     print(response.text)

# nmhc = pd.DataFrame(data)
# nmhc = nmhc[nmhc.sitename.isin(['桃園', '觀音', '大園', '平鎮', '龍潭'])]
# nmhc = nmhc[['sitename', 'monitordate', 'concentration']]

成功取得資料！


In [79]:
import requests
import pandas as pd

# 1. 基本設定
API_KEY = ""  # 替換為你的中央氣象署 API 授權碼
DATASET_ID = "O-A0001-001"      # 氣象觀測站-全測站逐時氣象資料
URL = f"https://opendata.cwa.gov.tw/api/v1/rest/datastore/{DATASET_ID}"

# 2. 設定 API 請求參數
params = {
    "Authorization": API_KEY,
    "format": "JSON"  # 回傳 JSON 格式
}

try:
    # 3. 發送 GET 請求
    response = requests.get(URL, params=params)
    response.raise_for_status()  # 若狀態碼非 200，拋出例外
    
    data = response.json()
    
    # 4. 解析回傳資料
    if data.get("success") == "true":
        stations = data["records"]["Station"]
        print(f"成功取得 {len(stations)} 個測站資料！\n")
        
        parsed_list = []
        for station in stations:
            # 基本資訊
            station_name = station.get("StationName")
            station_id = station.get("StationId")
            obs_time = station.get("ObsTime", {}).get("DateTime")
            
            # 地理位置
            geo_info = station.get("GeoInfo", {})
            county = geo_info.get("CountyName", "")
            town = geo_info.get("TownName", "")
            
            # 天氣要素 (WeatherElement)
            weather_elem = station.get("WeatherElement", {})
            weather = weather_elem.get("Weather")                 # 天氣現象
            air_temp = weather_elem.get("AirTemperature")         # 氣溫 (°C)
            rel_humidity = weather_elem.get("RelativeHumidity")   # 相對溼度 (%)
            wind_speed = weather_elem.get("WindSpeed")             # 風速 (m/s)
            
            parsed_list.append({
                "縣市": county,
                "鄉鎮區": town,
                "測站名稱": station_name,
                "測站代碼": station_id,
                "觀測時間": obs_time,
                "天氣現象": weather,
                "氣溫(°C)": air_temp,
                "相對溼度(%)": rel_humidity,
                "風速(m/s)": wind_speed
            })
            
        # 轉換為 DataFrame 方便預覽
        df = pd.DataFrame(parsed_list)
        
        # 印出前 10 筆資料
        print(df.head(10))
        
        # 若需要導出為 CSV 檔：
        # df.to_csv("hourly_weather.csv", index=False, encoding="utf-8-sig")
        
    else:
        print("API 回傳失敗，請檢查 API 授權碼或參數。")

except requests.exceptions.RequestException as e:
    print(f"網絡或 API 請求發生錯誤: {e}")

成功取得 874 個測站資料！

    縣市  鄉鎮區       測站名稱    測站代碼                       觀測時間  天氣現象 氣溫(°C)  \
0  花蓮縣  秀林鄉         崇德  C0TB40  2026-07-20T13:00:00+08:00    多雲   32.6   
1  臺東縣  卑南鄉         富山  C0SD20  2026-07-20T13:00:00+08:00     晴   31.6   
2  臺東縣  蘭嶼鄉       東清國小  C0SD30  2026-07-20T13:00:00+08:00     晴   31.4   
3  臺東縣  綠島鄉        牛頭山  C0SD40  2026-07-20T13:00:00+08:00     晴   31.6   
4  花蓮縣  秀林鄉         磐石  C0TC00  2026-07-20T13:00:00+08:00     陰    -99   
5  花蓮縣  萬榮鄉  西林林道12.5K  C0TC10  2026-07-20T13:00:00+08:00     陰   26.6   
6  花蓮縣  萬榮鄉    光復林道16K  C0TC20  2026-07-20T13:00:00+08:00  多雲有雨   23.1   
7  花蓮縣  萬榮鄉        虎頭山  C0TC30  2026-07-20T13:00:00+08:00    多雲    -99   
8  花蓮縣  秀林鄉        新白楊  C0TC40  2026-07-20T13:00:00+08:00     陰   20.9   
9  臺東縣  海端鄉       龍泉苗圃  C0SD90  2026-07-20T13:00:00+08:00    多雲   32.9   

  相對溼度(%) 風速(m/s)  
0      64     1.9  
1      70     2.3  
2      72     4.1  
3      70    10.0  
4     -99     -99  
5      89     2.8  
6      92     1.1  
7     -9

In [80]:
df = df[df['測站名稱'].isin(['桃園', '觀音', '大園', '平鎮', '龍潭', '竹圍'])]

In [81]:
df = df[['測站名稱', '觀測時間', '氣溫(°C)', '相對溼度(%)']]

In [82]:
df = df.rename(columns={'測站名稱': 'sitename', '觀測時間': 'publishtime', '氣溫(°C)': 'amb_temp', '相對溼度(%)': 'rh'})
df['sitename'] = df['sitename'] .replace('竹圍', '大園')

df

,sitename,publishtime,amb_temp,rh
98,大園,2026-07-20T13:00:00+08:00,30.3,75
251,平鎮,2026-07-20T13:00:00+08:00,-99,-99
258,龍潭,2026-07-20T13:00:00+08:00,32.0,54
597,桃園,2026-07-20T13:00:00+08:00,35.0,45
598,觀音,2026-07-20T13:00:00+08:00,33.8,59


In [87]:
df['publishtime'].str[0:-6]

98     2026-07-20T13:00:00
251    2026-07-20T13:00:00
258    2026-07-20T13:00:00
597    2026-07-20T13:00:00
598    2026-07-20T13:00:00
Name: publishtime, dtype: object

In [88]:
df['publishtime'] = df['publishtime'].str[0:-6]

In [89]:
df

,sitename,publishtime,amb_temp,rh
98,大園,2026-07-20T13:00:00,30.3,75
251,平鎮,2026-07-20T13:00:00,-99,-99
258,龍潭,2026-07-20T13:00:00,32.0,54
597,桃園,2026-07-20T13:00:00,35.0,45
598,觀音,2026-07-20T13:00:00,33.8,59


In [92]:
raw_data

,sitename,publishtime,pm2.5,pm10,o3,co,so2,no2,wind_speed,wind_direc
15,桃園,2026/07/20 13:00:00,7,10,46,0.14,0.1,5,2.2,266
16,大園,2026/07/20 13:00:00,1,10,34,0.1,0.3,2,4.1,250
17,觀音,2026/07/20 13:00:00,5,9,34,0.06,0.7,1,4.1,252
18,平鎮,2026/07/20 13:00:00,2,12,39,0.13,0.6,5,3,273
19,龍潭,2026/07/20 13:00:00,10,10,40,0.13,1.1,3,4.9,247


In [100]:
final_df = pd.merge(raw_data, df.drop('publishtime', axis=1), on=['sitename'], how='left')

In [101]:
final_df

,sitename,publishtime,pm2.5,pm10,o3,co,so2,no2,wind_speed,wind_direc,amb_temp,rh
0,桃園,2026/07/20 13:00:00,7,10,46,0.14,0.1,5,2.2,266,35.0,45
1,大園,2026/07/20 13:00:00,1,10,34,0.1,0.3,2,4.1,250,30.3,75
2,觀音,2026/07/20 13:00:00,5,9,34,0.06,0.7,1,4.1,252,33.8,59
3,平鎮,2026/07/20 13:00:00,2,12,39,0.13,0.6,5,3,273,-99,-99
4,龍潭,2026/07/20 13:00:00,10,10,40,0.13,1.1,3,4.9,247,32.0,54
